In [7]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import math
import pandas as pd


In [ ]:
from pathlib import Path

# auto-detect repo root by folder name
p = Path.cwd().resolve()
PROJECT_ROOT = next(parent for parent in [p] + list(p.parents) if parent.name == "graphCeylon-API")

print("CWD:", Path.cwd())
print("PROJECT_ROOT:", PROJECT_ROOT)

# ✅ correct input CSV in your repo
INPUT_CSV = PROJECT_ROOT / "data/recommender_service/clean/sltda_arrivals_monthly.csv"

# ✅ save outputs under models (recommended)
OUTPUT_DIR = PROJECT_ROOT / "models/recommender_service"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV = OUTPUT_DIR / "sltda_arrivals_lstm_forecast.csv"  # or any name you like

print("INPUT_CSV:", INPUT_CSV, "| exists:", INPUT_CSV.exists())
print("OUTPUT_CSV:", OUTPUT_CSV)


import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pandas.tseries.offsets import MonthBegin
from statsmodels.tsa.arima.model import ARIMA




New CWD: E:\Research_Final\component3_service


In [9]:
def load_arrivals() -> pd.DataFrame:
    df = pd.read_csv(INPUT_CSV, parse_dates=["date"])
    df = df.sort_values("date")
    df = df.set_index("date")

    # Ensure DatetimeIndex with monthly-start freq
    df.index = pd.to_datetime(df.index)
    df = df.asfreq("MS")   # 👈 Monthly Start frequency

    df["total_arrivals"] = df["total_arrivals"].astype(float)
    df.loc[df["total_arrivals"] < 1000, "total_arrivals"] = np.nan
    df["total_arrivals"] = df["total_arrivals"].interpolate(method="linear")

    print(f"Loaded {len(df)} monthly points from {df.index.min().date()} to {df.index.max().date()}")
    return df

df_arrivals = load_arrivals()


Loaded 94 monthly points from 2018-01-01 to 2025-10-01


In [ ]:
series = df_arrivals["total_arrivals"].values.astype(float).reshape(-1, 1)

from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler(feature_range=(0, 1))
series_scaled = scaler.fit_transform(series)  # (N,1)



In [11]:
WINDOW_SIZE = 24
VAL_MONTHS = 12

def create_sequences(data, window_size):
    """
    data: numpy array shape (N, 1) scaled
    returns X (num_samples, window_size, 1), y (num_samples, 1)
    """
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:i+window_size])
        y.append(data[i+window_size])
    return np.array(X), np.array(y)

X_all, y_all = create_sequences(series_scaled, WINDOW_SIZE)
print("X_all shape:", X_all.shape, "y_all shape:", y_all.shape)


X_all shape: (70, 24, 1) y_all shape: (70, 1)


In [12]:
# y_all length = len(series_scaled) - WINDOW_SIZE
# we want last VAL_MONTHS targets as validation
train_size = X_all.shape[0] - VAL_MONTHS

X_train = X_all[:train_size]
y_train = y_all[:train_size]
X_val   = X_all[train_size:]
y_val   = y_all[train_size:]

print("Train samples:", X_train.shape[0], "Val samples:", X_val.shape[0])


Train samples: 58 Val samples: 12


In [13]:
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = TimeSeriesDataset(X_train, y_train)
val_dataset   = TimeSeriesDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=16, shuffle=False)


In [14]:
import torch
import torch.nn as nn

# -------------------------------
# DEVICE SETUP
# -------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [15]:
class ArrivalLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=16, num_layers=1, dropout=0.0):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        last_out = out[:, -1, :]
        out = self.fc(last_out)
        return out

model = ArrivalLSTM(input_size=1, hidden_size=16, num_layers=1).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)  # slightly smaller LR


In [ ]:
# -------------------------------
# Reverse-scale metrics (helpers)
# -------------------------------

# The same scaler factor you used earlier:
# series_scaled = series / scale_factor
# make sure you have `scale_factor` available above

import numpy as np

def rmse_from_scaled(y_true_scaled, y_pred_scaled, scaler):
    y_true = scaler.inverse_transform(y_true_scaled)
    y_pred = scaler.inverse_transform(y_pred_scaled)
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def mae_from_scaled(y_true_scaled, y_pred_scaled, scaler):
    y_true = scaler.inverse_transform(y_true_scaled)
    y_pred = scaler.inverse_transform(y_pred_scaled)
    return np.mean(np.abs(y_true - y_pred))


In [ ]:
EPOCHS = 100
best_val_rmse = float("inf")
best_state_dict = None
patience = 15
no_improve = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_losses = []

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

    # validation
    model.eval()
    val_losses = []
    y_true_list = []
    y_pred_list = []

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            val_losses.append(loss.item())

            y_true_list.append(y_batch.cpu().numpy())
            y_pred_list.append(outputs.cpu().numpy())

    y_true_val = np.concatenate(y_true_list, axis=0)
    y_pred_val = np.concatenate(y_pred_list, axis=0)
    val_rmse = rmse_from_scaled(y_true_val, y_pred_val, scaler)
    val_mae  = mae_from_scaled(y_true_val, y_pred_val, scaler)


    # 🔥 early stopping logic
    if val_rmse < best_val_rmse - 1e-3:  # small tolerance
        best_val_rmse = val_rmse
        best_state_dict = model.state_dict()
        no_improve = 0
    else:
        no_improve += 1

    if epoch % 10 == 0 or epoch == 1:
        print(
            f"Epoch {epoch}/{EPOCHS} "
            f"- train_loss={np.mean(train_losses):.6f} "
            f"- val_loss={np.mean(val_losses):.6f} "
            f"- val_RMSE={val_rmse:.2f} "
            f"- val_MAE={val_mae:.2f}"
        )

    if no_improve >= patience:
        print(f"Early stopping at epoch {epoch} (no improvement for {patience} epochs).")
        break

# restore best weights
if best_state_dict is not None:
    model.load_state_dict(best_state_dict)
    print(f"Loaded best LSTM weights with val_RMSE={best_val_rmse:.2f}")


Epoch 1/100 - train_loss=0.159665 - val_loss=0.627900 - val_RMSE=400576.32 - val_MAE=392377.71
Epoch 10/100 - train_loss=0.108635 - val_loss=0.436625 - val_RMSE=334036.83 - val_MAE=324088.10
Epoch 20/100 - train_loss=0.051522 - val_loss=0.151133 - val_RMSE=196525.63 - val_MAE=178210.41
Epoch 30/100 - train_loss=0.044622 - val_loss=0.148978 - val_RMSE=195119.67 - val_MAE=176177.55
Epoch 40/100 - train_loss=0.039758 - val_loss=0.116588 - val_RMSE=172610.78 - val_MAE=150003.16
Epoch 50/100 - train_loss=0.035111 - val_loss=0.104742 - val_RMSE=163606.71 - val_MAE=138586.51
Epoch 60/100 - train_loss=0.033237 - val_loss=0.082306 - val_RMSE=145028.90 - val_MAE=120118.10
Epoch 70/100 - train_loss=0.029040 - val_loss=0.073775 - val_RMSE=137307.19 - val_MAE=113656.88
Epoch 80/100 - train_loss=0.024239 - val_loss=0.069514 - val_RMSE=133283.65 - val_MAE=110271.72
Epoch 90/100 - train_loss=0.017813 - val_loss=0.076706 - val_RMSE=140009.03 - val_MAE=117247.36
Early stopping at epoch 94 (no improvemen